# Conformal Prediction: Distribution-Free Uncertainty Guarantees

**Module 06 · AI Safety (ISO/PAS 8800)** · Notebook 5 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/06_AI_Safety_ISO_PAS_8800/05_conformal_prediction.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Intermediate/Advanced | [Uncertainty types](02_uncertainty_types.ipynb), [Calibration](04_calibration.ipynb) | ISO/PAS 8800:2024, ISO 21448:2022 (SOTIF) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

After working through this notebook you will be able to:

1. State the finite-sample coverage guarantee of **split conformal prediction** and explain why it is *distribution-free* and needs only **exchangeability**, not a correct model.
2. Build prediction **sets** for classification with two score functions — **LAC** (least-ambiguous, one minus the softmax probability of a class) and **APS** (adaptive) — and compare their set sizes.
3. Build prediction **intervals** for regression with **conformalised quantile regression (CQR)** and read them as a time-to-collision estimate.
4. Verify empirical coverage over many random splits, and see where **conditional** (class-wise) coverage fails — motivating **Mondrian/class-conditional** conformal.
5. Show that **distribution shift breaks exchangeability** with a synthetic "fog" shift, and recover coverage with **weighted** and **adaptive (ACI)** conformal prediction.
6. Connect prediction-set *size* to a conservative safety behaviour and to a **SOTIF / ISO/PAS 8800** assurance argument.
7. Conformalise a **real monocular distance estimate** on KITTI labels, check coverage per distance band and per class (Mondrian), and see coverage collapse for occluded and truncated objects.

The split-conformal threshold used here is identical to `avsafety.uq.conformal_threshold` in the repository.

## 1. Setup

Sections 2–8 use scikit-learn and NumPy on a small synthetic perception-like dataset, so that each property can be shown
in isolation; Section 9 repeats the key steps on real KITTI labels. Everything runs on CPU, and all generators are seeded.

> **Illustrative data:** the perception features in Sections 2–8 are synthetic, chosen for teaching; they are not measurements
> from a real sensor.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)
OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
CLASSES = ["background", "car", "pedestrian", "cyclist"]
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("Classes:", CLASSES)

## 2. Why conformal prediction?

A calibrated probability (Notebook 4) tells us that *on average* a "90% confident" prediction is right about 90% of the
time. A safety case usually needs something stronger and per-decision: a **set** of labels that contains the truth with a
guaranteed probability, whatever the data distribution and however wrong the model is.

**Split (inductive) conformal prediction** (Vovk et al., 2005; Angelopoulos & Bates, 2023) delivers exactly this. Given a
trained model, a held-out **calibration** set of $n$ labelled examples, and a target error rate $\alpha$:

1. Define a **nonconformity score** $s(x, y)$ — large when $(x,y)$ looks unusual for the model.
2. Compute calibration scores $s_i = s(x_i, y_i)$ and take the quantile
   $$\hat q = \text{the } \lceil (n+1)(1-\alpha) \rceil\text{-th smallest of } s_1,\dots,s_n.$$
3. For a new $x$, output the set $\mathcal C(x) = \{\,y : s(x,y) \le \hat q\,\}$.

Then, provided the calibration and test points are **exchangeable** (a weaker assumption than i.i.d.), the set satisfies
the **finite-sample** marginal guarantee

$$1-\alpha \;\le\; \Pr\big(Y_{\text{test}} \in \mathcal C(X_{\text{test}})\big) \;\le\; 1-\alpha + \tfrac{1}{n+1}.$$

No assumption is made about the model being correct or the distribution being Gaussian — hence *distribution-free*. The
price is paid in **set size**: a poor or uncertain model yields larger sets, which is itself an honest uncertainty signal.

In [ ]:
# A synthetic 4-class perception dataset: each object is described by features a detector might expose.
def make_perception(n, rng, fog=0.0):
    """Features: [bbox_area, aspect_ratio, speed, reflectivity, height]. 'fog' blurs the classes (covariate shift)."""
    n_per = np.array([0.35, 0.30, 0.20, 0.15])            # class priors: background, car, pedestrian, cyclist
    counts = rng.multinomial(n, n_per)
    feats, labels = [], []
    # per-class mean [bbox_area, aspect(w/h), speed, reflectivity, height]. Pedestrian and cyclist
    # deliberately overlap (similar height/area/reflectivity), so the classifier is genuinely unsure.
    mu = {0: [0.3, 1.0, 0.0, 0.3, 0.4], 1: [2.6, 1.5, 11.0, 0.6, 1.5],
          2: [0.7, 0.55, 1.5, 0.45, 1.7], 3: [1.0, 0.70, 3.8, 0.5, 1.6]}
    sd = {0: [0.80, 0.80, 1.6, 0.40, 0.64], 1: [1.92, 0.64, 9.6, 0.40, 0.64],
          2: [0.72, 0.48, 2.56, 0.40, 0.56], 3: [0.80, 0.56, 3.84, 0.40, 0.48]}
    for c, k in enumerate(counts):
        s = np.array(sd[c]) * (1.0 + 3.0 * fog)           # fog inflates feature noise
        feats.append(rng.normal(mu[c], s, size=(k, 5)))
        labels.append(np.full(k, c))
    X = np.vstack(feats); yv = np.concatenate(labels)
    p = rng.permutation(len(yv))
    return X[p], yv[p]

X_tr, y_tr = make_perception(4000, rng)
X_cal, y_cal = make_perception(2000, rng)
X_te, y_te = make_perception(4000, rng)
scaler = StandardScaler().fit(X_tr)
clf = MLPClassifier(hidden_layer_sizes=(32, 32), alpha=1e-3, max_iter=500, random_state=SEED)
clf.fit(scaler.transform(X_tr), y_tr)
acc = clf.score(scaler.transform(X_te), y_te)
print(f"Base classifier top-1 accuracy on clean test data: {acc:.3f}")
P_cal = clf.predict_proba(scaler.transform(X_cal))
P_te = clf.predict_proba(scaler.transform(X_te))

## 3. Split conformal for classification: LAC and APS

The simplest score is **LAC** (least-ambiguous set-valued classifier; Sadinle, Lei & Wasserman, 2019):
$s(x, y) = 1 - \hat p_y(x)$, the complement of the softmax probability of the candidate label. It produces the
*smallest* sets at a given coverage but can under-cover hard classes.

**APS** (adaptive prediction sets; Romano, Sesia & Candès, 2020) instead accumulates probabilities from the most likely
class downward until the true class is included: $s(x, y) = \sum_{y': \hat p_{y'} \ge \hat p_y} \hat p_{y'}$ (with a
randomised tie-break). APS gives larger but more *adaptive* sets — small where the model is confident, large where it is
unsure — which is often what a safety monitor wants.

The threshold formula is the one in `avsafety.uq.conformal_threshold`.

In [ ]:
def conformal_threshold(cal_scores, alpha):
    """The (n+1)(1-alpha)/n empirical quantile (Vovk et al.; identical to avsafety.uq.conformal_threshold)."""
    n = len(cal_scores)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(cal_scores, level, method="higher"))

def lac_scores(P, y):
    return 1.0 - P[np.arange(len(y)), y]

def aps_scores(P, y, rng, randomize=True):
    """Adaptive prediction-set score (Romano et al., 2020) with randomised tie-break."""
    order = np.argsort(-P, axis=1)
    sorted_p = np.take_along_axis(P, order, axis=1)
    cum = np.cumsum(sorted_p, axis=1)
    rank = np.argmax(order == y[:, None], axis=1)                 # position of the true label
    cum_at = cum[np.arange(len(y)), rank]
    p_at = sorted_p[np.arange(len(y)), rank]
    u = rng.random(len(y)) if randomize else 1.0
    return cum_at - u * p_at

def aps_sets(P, q, rng, randomize=True):
    order = np.argsort(-P, axis=1)
    sorted_p = np.take_along_axis(P, order, axis=1)
    cum = np.cumsum(sorted_p, axis=1)
    u = rng.random((len(P), 1)) if randomize else 1.0
    keep = (cum - u * sorted_p) <= q
    keep[:, 0] = True                                            # always include the top class
    sets = np.zeros_like(P, dtype=bool)
    np.put_along_axis(sets, order, keep, axis=1)
    return sets

ALPHA = 0.1
q_lac = conformal_threshold(lac_scores(P_cal, y_cal), ALPHA)
lac_sets = (1.0 - P_te) <= q_lac

rng_aps = np.random.default_rng(SEED)
q_aps = conformal_threshold(aps_scores(P_cal, y_cal, rng_aps), ALPHA)
aps_set = aps_sets(P_te, q_aps, np.random.default_rng(SEED + 1))

def report(sets, y, name):
    cov = sets[np.arange(len(y)), y].mean()
    size = sets.sum(1).mean()
    print(f"{name:6s}: threshold applied, marginal coverage {cov:.3f} (target {1-ALPHA:.2f}), "
          f"mean set size {size:.2f}")
    return cov, size

report(lac_sets, y_te, "LAC")
report(aps_set, y_te, "APS")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
sizes_lac = lac_sets.sum(1); sizes_aps = aps_set.sum(1)
ax = axes[0]
w = 0.35; vals = np.arange(1, 5)
ax.bar(vals - w/2, [np.mean(sizes_lac == k) for k in vals], w, color=OKABE_ITO[1], label="LAC")
ax.bar(vals + w/2, [np.mean(sizes_aps == k) for k in vals], w, color=OKABE_ITO[5], label="APS")
ax.set_xlabel("Prediction-set size |C(x)|"); ax.set_ylabel("Fraction of test objects")
ax.set_xticks(vals); ax.set_title("Set-size distribution at 90% coverage"); ax.legend()

ax = axes[1]
conf = P_te.max(1)
bins = np.linspace(0.25, 1.0, 10)
idx = np.digitize(conf, bins)
centres = 0.5 * (bins[:-1] + bins[1:])
m_lac = [sizes_lac[idx == i + 1].mean() if (idx == i + 1).any() else np.nan for i in range(len(centres))]
m_aps = [sizes_aps[idx == i + 1].mean() if (idx == i + 1).any() else np.nan for i in range(len(centres))]
ax.plot(centres, m_lac, "o-", color=OKABE_ITO[1], label="LAC")
ax.plot(centres, m_aps, "s-", color=OKABE_ITO[5], label="APS")
ax.set_xlabel("Top-1 softmax confidence"); ax.set_ylabel("Mean set size")
ax.set_title("APS sets grow where the model is unsure"); ax.legend()
plt.tight_layout(); plt.show()
print("Both reach the coverage target; APS trades a little extra average size for adaptivity to difficulty.")

## 4. Verifying the guarantee: coverage over many random splits

The guarantee is about the randomness of the calibration/test split, so a single split's coverage is itself random. Over
many random splits the coverage concentrates around $1-\alpha$; theory (Vovk, 2012) predicts it follows a
$\text{Beta}(n+1-l, l)$ distribution with $l = \lfloor (n+1)\alpha \rfloor$ and mean slightly above $1-\alpha$. We draw
200 splits and histogram the coverage — it should sit around 0.90, not below.

In [ ]:
def one_split_coverage(P, y, alpha, rng):
    n = len(y); perm = rng.permutation(n); half = n // 2
    cal, te = perm[:half], perm[half:]
    q = conformal_threshold(lac_scores(P[cal], y[cal]), alpha)
    sets = (1.0 - P[te]) <= q
    return sets[np.arange(len(te)), y[te]].mean()

P_all = clf.predict_proba(scaler.transform(np.vstack([X_cal, X_te])))
y_all = np.concatenate([y_cal, y_te])
rng_cov = np.random.default_rng(SEED)
covs = np.array([one_split_coverage(P_all, y_all, ALPHA, rng_cov) for _ in range(200)])

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(covs, bins=25, color=OKABE_ITO[1], alpha=0.85, edgecolor="white")
ax.axvline(1 - ALPHA, color=OKABE_ITO[5], lw=2, label=f"target 1-alpha = {1-ALPHA:.2f}")
ax.axvline(covs.mean(), color=OKABE_ITO[2], ls="--", lw=2, label=f"mean coverage = {covs.mean():.3f}")
ax.set_xlabel("Empirical marginal coverage"); ax.set_ylabel("Number of random splits")
ax.set_title("Coverage over 200 calibration/test splits concentrates at 1-alpha")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Mean coverage over splits {covs.mean():.3f} (theory: marginal coverage is >= {1-ALPHA:.2f} in "
      f"expectation); individual splits range {covs.min():.3f} to {covs.max():.3f}.")

## 5. Where marginal coverage is not enough: conditional coverage and Mondrian conformal

The marginal guarantee averages over **all** inputs. It says nothing about any particular group — and a pedestrian-class
coverage of 80% hidden inside a 90% average would be unacceptable. This is *conditional* coverage, which split conformal
does **not** guarantee. We can see the gap by measuring coverage per true class, then fix it with **Mondrian / class-
conditional** conformal prediction (Vovk et al., 2005): calibrate a **separate threshold within each class**. Each group
then gets its own finite-sample guarantee, at the cost of larger sets for the harder classes.

In [ ]:
# marginal LAC coverage per class (one shared threshold q_lac from Section 3)
cov_marg = [lac_sets[y_te == c][np.arange((y_te == c).sum()), c].mean() for c in range(4)]

# Mondrian: a separate threshold per calibration class
q_class = {c: conformal_threshold(lac_scores(P_cal[y_cal == c], y_cal[y_cal == c]), ALPHA) for c in range(4)}
mond_sets = np.zeros_like(P_te, dtype=bool)
for c in range(4):                                           # include class c if its own threshold admits it
    mond_sets[:, c] = (1.0 - P_te[:, c]) <= q_class[c]
mond_sets[np.arange(len(P_te)), P_te.argmax(1)] = True       # never return an empty set
cov_mond = [mond_sets[y_te == c][np.arange((y_te == c).sum()), c].mean() for c in range(4)]

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(4); w = 0.38
ax.bar(x - w/2, cov_marg, w, color=OKABE_ITO[1], label="marginal LAC")
ax.bar(x + w/2, cov_mond, w, color=OKABE_ITO[2], label="Mondrian (class-conditional)")
ax.axhline(1 - ALPHA, color=OKABE_ITO[5], lw=2, ls="--", label=f"target {1-ALPHA:.2f}")
ax.set_xticks(x, CLASSES); ax.set_ylim(0.7, 1.0)
ax.set_ylabel("Coverage for the true class"); ax.set_xlabel("True class")
ax.set_title("Marginal coverage can fail per class; Mondrian restores it"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
worst = CLASSES[int(np.argmin(cov_marg))]
print(f"Worst class under marginal conformal: {worst} ({min(cov_marg):.3f}); "
      f"under Mondrian all classes reach about {np.mean(cov_mond):.3f}.")
print(f"Mean set size rises from {lac_sets.sum(1).mean():.2f} (marginal) to {mond_sets.sum(1).mean():.2f} (Mondrian).")

## 6. Conformal regression: calibrated distance intervals with CQR

Perception and planning also produce **continuous** estimates — object distance, time-to-collision, trajectory position —
and a safety monitor wants a *calibrated interval*, not a point estimate. We take **distance estimation** from a range
sensor: the measurement noise grows with range (far objects are noisier), so the ideal interval is **heteroscedastic**.
Naive conformal regression adds a constant $\pm \hat q$ band everywhere, which is too wide for near objects and too narrow
for far ones.

**Conformalised quantile regression (CQR)** (Romano, Patterson & Candès, 2019) fixes this. Train two quantile regressors
for the lower and upper quantiles $\hat q_{\alpha/2}, \hat q_{1-\alpha/2}$. On the calibration set compute the
conformity score

$$E_i = \max\{\hat q_{\alpha/2}(x_i) - y_i,\; y_i - \hat q_{1-\alpha/2}(x_i)\},$$

take its $\lceil(n+1)(1-\alpha)\rceil/n$ quantile $\hat Q$, and output the interval
$[\hat q_{\alpha/2}(x) - \hat Q,\; \hat q_{1-\alpha/2}(x) + \hat Q]$. The interval **adapts** its width to local
difficulty yet keeps the finite-sample coverage guarantee.

In [ ]:
# Synthetic distance regression: measurement noise grows with true range (heteroscedastic).
def make_distance(n, rng):
    d = rng.uniform(5, 60, n)                          # true distance [m]
    noise_sd = 0.4 + 0.09 * d                          # far objects are noisier
    rng_meas = d + rng.normal(0, noise_sd, n)          # noisy range reading
    intensity = np.clip(1.2 / (1 + (d / 25) ** 2) + rng.normal(0, 0.05, n), 0, None)
    X = np.column_stack([rng_meas, intensity, rng_meas * intensity])
    return X, d

Xr_tr, yr_tr = make_distance(3000, rng)
Xr_cal, yr_cal = make_distance(2000, rng)
Xr_te, yr_te = make_distance(2000, rng)

lo = GradientBoostingRegressor(loss="quantile", alpha=ALPHA/2, n_estimators=300, max_depth=3, random_state=SEED).fit(Xr_tr, yr_tr)
hi = GradientBoostingRegressor(loss="quantile", alpha=1-ALPHA/2, n_estimators=300, max_depth=3, random_state=SEED).fit(Xr_tr, yr_tr)

def cqr_interval(X, Q):
    return lo.predict(X) - Q, hi.predict(X) + Q

# conformity score on calibration, then threshold (same quantile rule as the classifier)
E = np.maximum(lo.predict(Xr_cal) - yr_cal, yr_cal - hi.predict(Xr_cal))
Q = conformal_threshold(E, ALPHA)

# baseline: naive conformal with a point regressor +/- constant band
point = GradientBoostingRegressor(n_estimators=300, max_depth=3, random_state=SEED).fit(Xr_tr, yr_tr)
res = np.abs(yr_cal - point.predict(Xr_cal))
q_naive = conformal_threshold(res, ALPHA)

L_cqr, U_cqr = cqr_interval(Xr_te, Q)
cov_cqr = np.mean((yr_te >= L_cqr) & (yr_te <= U_cqr)); w_cqr = np.mean(U_cqr - L_cqr)
L_nv, U_nv = point.predict(Xr_te) - q_naive, point.predict(Xr_te) + q_naive
cov_nv = np.mean((yr_te >= L_nv) & (yr_te <= U_nv)); w_nv = np.mean(U_nv - L_nv)
print(f"CQR   : coverage {cov_cqr:.3f} (target {1-ALPHA:.2f}), mean interval width {w_cqr:.2f} m")
print(f"Naive : coverage {cov_nv:.3f} (target {1-ALPHA:.2f}), mean interval width {w_nv:.2f} m")

In [ ]:
d_true = yr_te
order = np.argsort(d_true)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
ax = axes[0]
ax.fill_between(d_true[order], L_nv[order], U_nv[order], color=OKABE_ITO[0], alpha=0.35, label="naive (constant width)")
ax.fill_between(d_true[order], L_cqr[order], U_cqr[order], color=OKABE_ITO[1], alpha=0.5, label="CQR (adaptive)")
ax.plot([5, 60], [5, 60], color=OKABE_ITO[7], lw=0.8, ls=":", label="ideal estimate")
ax.set_xlabel("True distance [m]"); ax.set_ylabel("Estimated distance interval [m]")
ax.set_title("CQR narrows for near objects, widens for far ones"); ax.legend(fontsize=8)

ax = axes[1]
bins = np.linspace(5, 60, 7)
bc = 0.5 * (bins[:-1] + bins[1:])
idx = np.digitize(d_true, bins)
cov_cqr_d = [((yr_te >= L_cqr) & (yr_te <= U_cqr))[idx == i].mean() for i in range(1, len(bins))]
cov_nv_d = [((yr_te >= L_nv) & (yr_te <= U_nv))[idx == i].mean() for i in range(1, len(bins))]
ax.plot(bc, cov_cqr_d, "o-", color=OKABE_ITO[1], label="CQR")
ax.plot(bc, cov_nv_d, "s-", color=OKABE_ITO[0], label="naive (constant width)")
ax.axhline(1 - ALPHA, color=OKABE_ITO[5], ls="--", label=f"target {1-ALPHA:.2f}")
ax.set_xlabel("True distance [m]"); ax.set_ylabel("Conditional coverage")
ax.set_ylim(0.7, 1.02); ax.set_title("Naive under-covers far objects; CQR stays near target"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"Naive coverage at the farthest bin {cov_nv_d[-1]:.3f} (dangerously low) vs CQR {cov_cqr_d[-1]:.3f}; "
      f"CQR even has the smaller mean width ({w_cqr:.1f} m vs {w_nv:.1f} m) because it does not over-pad near objects.")

## 7. Distribution shift breaks the guarantee

The coverage guarantee rests on **exchangeability**: calibration and test data must be interchangeable. Operational
reality — a sensor degrading in fog, a new city, a software update — breaks it. We calibrate on clean weather and test
under a synthetic **fog** shift (feature noise inflated by the `fog` parameter) and watch coverage fall below the target.
This is the conformal-prediction face of a **SOTIF** triggering condition: the guarantee silently lapses exactly when it
is needed most.

In [ ]:
q_clean = conformal_threshold(lac_scores(P_cal, y_cal), ALPHA)
fogs = np.linspace(0.0, 1.0, 6)
cov_shift = []
rng_fog = np.random.default_rng(SEED)
for fg in fogs:
    Xf, yf = make_perception(4000, rng_fog, fog=fg)
    Pf = clf.predict_proba(scaler.transform(Xf))
    sets = (1.0 - Pf) <= q_clean
    cov_shift.append(sets[np.arange(len(yf)), yf].mean())

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(fogs, cov_shift, "o-", color=OKABE_ITO[5], lw=2, label="coverage under fog (fixed clean threshold)")
ax.axhline(1 - ALPHA, color=OKABE_ITO[2], ls="--", lw=2, label=f"target {1-ALPHA:.2f}")
ax.set_xlabel("Fog severity (feature-noise multiplier)"); ax.set_ylabel("Marginal coverage")
ax.set_title("Covariate shift breaks exchangeability -> coverage drops below target")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Coverage falls from {cov_shift[0]:.3f} (no fog) to {cov_shift[-1]:.3f} (heavy fog) while the model keeps")
print("emitting the same-looking sets -- a silent loss of the guarantee.")

### Restoring coverage: weighted and adaptive conformal

Two remedies, matching the two situations in practice:

* **Weighted conformal** (Tibshirani, Barber, Candès & Ramdas, 2019) handles a *known covariate shift* — the inputs are drawn from a different $P(X)$ but the relationship $P(Y\mid X)$ is unchanged. Reweight each calibration score by the likelihood ratio $w(x) = \mathrm{d}P_{\text{test}}/\mathrm{d}P_{\text{cal}}(x)$ before taking the quantile; a weighted exchangeability replaces the plain one and coverage is restored.
* **Adaptive conformal inference (ACI)** (Gibbs & Candès, 2021) handles *unknown, time-varying* shift online: adjust the effective level $\alpha_t$ after every observation by $\alpha_{t+1} = \alpha_t + \gamma(\alpha - \mathrm{err}_t)$, where $\mathrm{err}_t$ is 1 if the last set missed. It needs no model of the shift and drives the long-run error rate to $\alpha$ automatically.

The fog shift above also changes $P(Y\mid X)$ (the model simply errs more), which **no** covariate-shift reweighting can
repair — an important caveat. So we demonstrate weighted conformal on a clean *covariate* shift (an exponential tilt that
over-samples low-confidence inputs), and ACI on the fog stream where the shift is unknown and grows over time.

In [ ]:
def weighted_threshold(scores, weights, alpha):
    order = np.argsort(scores); s = scores[order]; w = weights[order]
    w = np.r_[w, weights.mean()]; s = np.r_[s, np.inf]       # +inf point mass for the test weight
    cw = np.cumsum(w) / w.sum()
    return float(s[np.searchsorted(cw, 1 - alpha)])

# a genuine covariate shift: draw a test set that over-samples hard (low-confidence) inputs.
BETA = 6.0
rng_w = np.random.default_rng(SEED + 5)
X_pool, y_pool = make_perception(20000, rng_w)
P_pool = clf.predict_proba(scaler.transform(X_pool))
w_pool = np.exp(-BETA * P_pool.max(1))                       # known tilt weight w(x), function of x only
sel = rng_w.choice(len(y_pool), 4000, replace=False, p=w_pool / w_pool.sum())
P_sh, y_sh = P_pool[sel], y_pool[sel]

sc_cal = lac_scores(P_cal, y_cal)
w_cal = np.exp(-BETA * P_cal.max(1))                         # same tilt applied to calibration scores
q_w = weighted_threshold(sc_cal, w_cal, ALPHA)
cov_unweighted = ((1.0 - P_sh)[np.arange(len(y_sh)), y_sh] <= q_clean).mean()
cov_weighted = ((1.0 - P_sh)[np.arange(len(y_sh)), y_sh] <= q_w).mean()
print(f"Covariate shift (over-sampling hard inputs):")
print(f"  unweighted conformal coverage {cov_unweighted:.3f} (below target) -> weighted {cov_weighted:.3f} "
      f"(restored towards {1-ALPHA:.2f}).")

In [ ]:
# ACI on a time-ramping fog stream
rng_aci = np.random.default_rng(SEED + 9)
T = 3000
fog_t = np.clip(np.linspace(-0.3, 1.2, T), 0, 1.2)
alpha_t = ALPHA; gamma = 0.02
a_fixed = ALPHA
err_aci, err_fixed, alphas = [], [], []
for t in range(T):
    Xt, yt = make_perception(1, rng_aci, fog=fog_t[t])
    Pt = clf.predict_proba(scaler.transform(Xt))[0]
    q_t = conformal_threshold(lac_scores(P_cal, y_cal), min(max(alpha_t, 1e-3), 0.999))
    covered = (1.0 - Pt[yt[0]]) <= q_t
    err_aci.append(0 if covered else 1)
    q_fx = conformal_threshold(lac_scores(P_cal, y_cal), a_fixed)
    err_fixed.append(0 if (1.0 - Pt[yt[0]]) <= q_fx else 1)
    alpha_t = alpha_t + gamma * (ALPHA - (0 if covered else 1))
    alphas.append(alpha_t)

win = 200
roll = lambda e: np.convolve(e, np.ones(win) / win, "valid")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
ax = axes[0]
ax.plot(roll(err_fixed), color=OKABE_ITO[5], label="fixed alpha (clean calibration)")
ax.plot(roll(err_aci), color=OKABE_ITO[2], label="ACI (adaptive)")
ax.axhline(ALPHA, color="black", ls="--", label=f"target error {ALPHA}")
ax.set_xlabel("Time step"); ax.set_ylabel(f"Rolling error rate ({win}-step)")
ax.set_title("ACI tracks the target error as fog ramps up"); ax.legend(fontsize=8)
ax = axes[1]
ax.plot(alphas, color=OKABE_ITO[2]); ax.axhline(ALPHA, color="black", ls="--")
ax.set_xlabel("Time step"); ax.set_ylabel("Effective alpha_t")
ax.set_title("ACI raises its effective alpha to stay covered")
plt.tight_layout(); plt.show()
print(f"Long-run error: fixed {np.mean(err_fixed):.3f} (exceeds {ALPHA} once fog sets in), "
      f"ACI {np.mean(err_aci):.3f} (kept much closer to the target).")

## 8. Using set size in a safety function, and the link to ISO/PAS 8800

The most useful output for a safety architecture is not the label but the **set**. A singleton `{car}` is an unambiguous
decision; a set `{pedestrian, cyclist}` is a *calibrated statement of ambiguity* with a guaranteed chance of containing
the truth. A safety monitor can act on the set itself:

* **Singleton** → act normally on the predicted class.
* **Ambiguous set containing a vulnerable road user** → take the conservative action for the most protected member (treat a `{pedestrian, cyclist}` set as a pedestrian: reduce speed, widen margin).
* **Large or empty set** → the input is out of the model's competence; hand over or enter a minimal-risk condition.

This turns a distribution-free statistical guarantee into an auditable decision rule.

In [ ]:
def safety_action(sets):
    """Map each prediction set to a driving action. Conservative towards vulnerable road users."""
    vru = {CLASSES.index("pedestrian"), CLASSES.index("cyclist")}
    actions = []
    for row in sets:
        members = set(np.flatnonzero(row))
        if len(members) == 0:
            actions.append("handover / minimal-risk (empty set)")
        elif len(members) == 1:
            actions.append("normal (confident)")
        elif members & vru:
            actions.append("conservative (VRU possible)")
        else:
            actions.append("caution (ambiguous, no VRU)")
    return np.array(actions)

acts = safety_action(aps_set)
dist = pd.Series(acts).value_counts(normalize=True).sort_index()
# of the objects that are truly a VRU, how many trigger a conservative/handover action?
truth_vru = np.isin(y_te, [CLASSES.index("pedestrian"), CLASSES.index("cyclist")])
protective = np.isin(acts, ["conservative (VRU possible)", "handover / minimal-risk (empty set)"])
recall_vru = protective[truth_vru].mean()

fig, ax = plt.subplots(figsize=(8, 3.4))
colors = {"normal (confident)": OKABE_ITO[2], "caution (ambiguous, no VRU)": OKABE_ITO[0],
          "conservative (VRU possible)": OKABE_ITO[5], "handover / minimal-risk (empty set)": OKABE_ITO[7]}
ax.barh(dist.index, dist.values, color=[colors[k] for k in dist.index])
for i, v in enumerate(dist.values):
    ax.text(v + 0.005, i, f"{v:.1%}", va="center")
ax.set_xlabel("Fraction of test objects"); ax.set_title("Driving actions derived from APS prediction sets")
plt.tight_layout(); plt.show()
print(f"{recall_vru:.1%} of true pedestrians/cyclists trigger a protective action via the set rule "
      f"(vs the {1-ALPHA:.0%} coverage floor the set guarantee provides).")

**Relation to ISO/PAS 8800 and SOTIF.** ISO/PAS 8800:2024 asks for *quantified, validated* measures of AI performance
and residual uncertainty as part of the AI safety argument, and ISO 21448 (SOTIF) is concerned precisely with
performance insufficiencies and triggering conditions. Conformal prediction contributes three things to such an argument:

1. a **distribution-free coverage guarantee** that can be stated as a verifiable requirement ("the prediction set contains the true class with probability at least $1-\alpha$ on the operational distribution");
2. **set size / interval width** as a continuous, monitorable measure of residual uncertainty that a runtime monitor can threshold;
3. an explicit, testable **failure mode** — coverage loss under distribution shift — that maps directly onto a SOTIF triggering condition and can be mitigated with weighted or adaptive conformal, with the mitigation itself evidenced by the empirical-coverage experiments above.

The guarantee is only as good as the **exchangeability** assumption, so the safety case must bound the operational
shift, which is where ISO 21448's scenario and ODD analysis and this module's monitoring notebooks come back in.

## 9. Real data: conformal distance intervals for a monocular camera on KITTI

The sections above used synthetic features so that every property could be controlled. Here the same machinery is applied to a real perception quantity that an AEB function consumes: the **distance to an object estimated from a single camera image**.

A monocular camera has no direct range measurement. By the pinhole model, an object of height $H$ at distance $d$ appears $h \approx f H / d$ pixels tall, so distance can be inferred from the 2D box (its height, width and the image row of its lower edge, which encodes the ground-contact point on a flat road). We train a gradient-boosted regressor on $\log d$ from these box features and the class, and compare it with the **ground-truth distance** $\sqrt{x^2+z^2}$ of the 3D box centre in KITTI's camera coordinates (annotated with the help of the LiDAR). Then we conformalise it.

### Data and licence

All KITTI object-detection training labels (one 5 MB archive, no images; Geiger et al., 2012), licence CC BY-NC-SA 3.0 (non-commercial use). We keep cars, pedestrians and cyclists that are clearly visible (KITTI difficulty Easy or Moderate: box at least 25 px tall, truncation ≤ 30 %, fully or partly visible) and split **by frame** into training (50 %), calibration (25 %) and test (25 %) frames, so that no image contributes to two splits.

**Caveat.** The 2D boxes are the annotators' boxes, i.e. an idealised detector. A real detector's box errors add to the distance error, and intervals calibrated on annotated boxes would under-cover detector boxes; calibrate on the boxes of the detector you deploy.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

pd.set_option("display.width", 180)
print(D.attribution("KITTI"))
try:
    labels = D.kitti_object_labels_all()
except D.DataUnavailable as exc:
    raise SystemExit(f"KITTI labels could not be downloaded ({exc}). Run the notebook with internet access.")
KCLS = ["Car", "Pedestrian", "Cyclist"]
obj = labels[labels.type.isin(KCLS)].copy()
obj["fid"] = obj.frame.astype(int)
obj["log_h"] = np.log((obj.y2 - obj.y1).clip(lower=1)); obj["log_w"] = np.log((obj.x2 - obj.x1).clip(lower=1))
obj["u_centre"] = (obj.x1 + obj.x2) / 2
for k in KCLS:
    obj["is_" + k] = (obj.type == k).astype(float)
FEATURES = ["log_h", "log_w", "y2", "u_centre"] + ["is_" + k for k in KCLS]   # what a 2D detector provides

frame_ids = np.array(sorted(obj.fid.unique()))
np.random.default_rng(SEED).shuffle(frame_ids)
n_f = len(frame_ids)
split_of = {f: ("train" if i < n_f // 2 else "cal" if i < 3 * n_f // 4 else "test") for i, f in enumerate(frame_ids)}
obj["split"] = obj.fid.map(split_of)
evaluable = obj[obj.difficulty.isin(["Easy", "Moderate"])]
tr_k, cal_k, te_k = (evaluable[evaluable.split == s] for s in ("train", "cal", "test"))

with threadpool_limits(limits=1):          # single thread: deterministic and fast enough for 14k samples
    dist_model = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(tr_k[FEATURES], np.log(tr_k.distance_m))
predict_d = lambda d: np.exp(dist_model.predict(d[FEATURES]))

print(f"Objects (train / calibration / test): {len(tr_k):,} / {len(cal_k):,} / {len(te_k):,} "
      f"from {n_f:,} frames; classes in test: {te_k.type.value_counts().to_dict()}")
err = predict_d(te_k) - te_k.distance_m
print(f"Point estimate on test objects: median |error| {np.median(np.abs(err)):.2f} m, "
      f"median relative error {np.median(np.abs(err) / te_k.distance_m):.1%}")

### 9.1 Two nonconformity scores: metres or ratio?

- **Absolute residual** $s = |d - \hat d|$ gives a constant band $\hat d \pm \hat q$ (in metres), like the naive band of Section 6.
- **Log residual** $s = |\log d - \log \hat d|$ gives a multiplicative band $[\hat d\,e^{-\hat q},\ \hat d\,e^{\hat q}]$. Because $\delta d / d \approx \delta h / h$ for a pinhole camera, the error of a monocular estimate grows roughly in proportion to distance, and a relative band follows that (it is a special case of the normalised scores of Lei et al., 2018).

Both use the same $\lceil (n+1)(1-\alpha)\rceil / n$ quantile as `avsafety.uq.conformal_threshold`, checked numerically below. The marginal guarantee holds for both; they differ in **where** the misses fall.

In [ ]:
from avsafety import uq
d_cal, d_te = predict_d(cal_k), predict_d(te_k)
s_abs = np.abs(cal_k.distance_m.values - d_cal)
s_log = np.abs(np.log(cal_k.distance_m.values) - np.log(d_cal))
q_abs, q_log = conformal_threshold(s_abs, ALPHA), conformal_threshold(s_log, ALPHA)
# Same order statistic as the toolkit: feed the scores as 1 - p(true class) of a one-class problem
assert np.isclose(q_abs, uq.conformal_threshold((1 - s_abs)[:, None], np.zeros(len(s_abs), int), ALPHA))

y_te = te_k.distance_m.values
intervals = {"absolute (± metres)": (d_te - q_abs, d_te + q_abs),
             "log (× / ÷ factor)": (d_te * np.exp(-q_log), d_te * np.exp(q_log))}
BANDS = [0, 10, 20, 30, 40, 80]
band = pd.cut(y_te, BANDS)
print(f"Calibration n = {len(s_abs):,}: q_abs = {q_abs:.2f} m (band ±{q_abs:.2f} m); "
      f"q_log = {q_log:.3f} (factor {np.exp(q_log):.3f}, i.e. about ±{(np.exp(q_log) - 1):.0%})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
centres = [f"{a}-{b}" for a, b in zip(BANDS[:-1], BANDS[1:])]
for (name, (lo_, hi_)), col, mk in zip(intervals.items(), [OKABE_ITO[0], OKABE_ITO[4]], ["s", "o"]):
    cov = (y_te >= lo_) & (y_te <= hi_)
    by = pd.DataFrame({"cov": cov, "w": hi_ - lo_, "band": band}).groupby("band", observed=False)
    axes[0].plot(centres, by["cov"].mean().values, mk + "-", color=col, lw=2, label=f"{name}: overall {cov.mean():.3f}")
    axes[1].plot(centres, by["w"].mean().values, mk + "-", color=col, lw=2, label=name)
axes[0].axhline(1 - ALPHA, color=OKABE_ITO[5], ls="--", label=f"target {1 - ALPHA:.2f}")
axes[0].set_xlabel("True distance band [m]"); axes[0].set_ylabel("Coverage on test objects"); axes[0].set_ylim(0.5, 1.02)
axes[0].set_title("Same marginal coverage, different conditional coverage"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("True distance band [m]"); axes[1].set_ylabel("Mean interval width [m]")
axes[1].set_title("Interval width by distance"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Objects per band in the test set:", dict(zip(centres, np.bincount(band.codes[band.codes >= 0], minlength=len(centres)).tolist())))
for name, (lo_, hi_) in intervals.items():
    cov = pd.Series((y_te >= lo_) & (y_te <= hi_)).groupby(band, observed=False).mean()
    print(f"{name:22s}: coverage {cov.iloc[0]:.3f} in the nearest band, {cov.iloc[-1]:.3f} in the farthest band")

### 9.2 Mondrian conformal by class and by predicted distance

Pedestrians are the class the AEB pedestrian sub-scope is about, but they are a minority of the labels; a marginal guarantee dominated by cars says little about them. **Mondrian** conformal (Section 5) calibrates a separate threshold in each group. The grouping must be computable at test time, so we can group by the **detected class** and by the **predicted** distance band, never by the true distance.

In [ ]:
def mondrian_log(groups_cal, groups_te):
    """Per-group log-residual threshold; returns test intervals and the thresholds."""
    q = {g: conformal_threshold(s_log[groups_cal == g], ALPHA) for g in np.unique(groups_cal)}
    qt = np.array([q[g] for g in groups_te])
    return (d_te * np.exp(-qt), d_te * np.exp(qt)), q

pred_band = lambda d: np.digitize(d, [15, 30])                       # 0: < 15 m, 1: 15-30 m, 2: > 30 m (predicted)
variants = {
    "marginal (one threshold)": intervals["log (× / ÷ factor)"],
    "Mondrian by class": mondrian_log(cal_k.type.values, te_k.type.values)[0],
    "Mondrian by class x predicted band": mondrian_log(
        cal_k.type.values + "/" + pred_band(d_cal).astype(str), te_k.type.values + "/" + pred_band(d_te).astype(str))[0],
}
q_class = mondrian_log(cal_k.type.values, te_k.type.values)[1]
rows = []
for name, (lo_, hi_) in variants.items():
    cov = (y_te >= lo_) & (y_te <= hi_)
    row = {"variant": name, "overall": cov.mean()}
    row.update({k: cov[te_k.type.values == k].mean() for k in KCLS})
    row.update({f"{c} m": v for c, v in zip(centres, pd.Series(cov).groupby(band, observed=False).mean().values)})
    row["mean width [m]"] = np.mean(hi_ - lo_)
    rows.append(row)
table = pd.DataFrame(rows).set_index("variant")
print(f"Test coverage at target {1 - ALPHA:.2f} per class and per TRUE distance band, and mean width")
print(table.round(3).to_string())
print("Per-class log thresholds: " + ", ".join(f"{k} ×/÷ {np.exp(v):.3f}" for k, v in q_class.items()))

### 9.3 A real exchangeability break: truncated and occluded objects

The calibration data contained only clearly visible objects. An object **truncated at the image border** violates the model's premise: the box ends at the image edge, so it is smaller than the object and the distance is overestimated. Truncation is typical of objects close to the vehicle and to the side, such as a pedestrian entering the field of view from the kerb or a vehicle cutting in, i.e. exactly where an AEB decision is imminent. **Largely occluded** objects (a pedestrian behind a parked van) are a second candidate. Below, the intervals calibrated on visible objects are applied unchanged to truncated and occluded objects (boxes still ≥ 25 px tall) from the same test frames.

In [ ]:
hard_k = obj[~obj.difficulty.isin(["Easy", "Moderate"]) & ((obj.y2 - obj.y1) >= 25) & (obj.split == "test")].copy()
d_h = predict_d(hard_k)
qh = hard_k.type.map(q_class).values
hard_k["covered"] = (hard_k.distance_m >= d_h * np.exp(-qh)) & (hard_k.distance_m <= d_h * np.exp(qh))
hard_k["overestimated"] = d_h > hard_k.distance_m
hard_k["reason"] = np.select([hard_k.truncated > 0.3, hard_k.occluded == 2], ["truncated > 30 %", "largely occluded"],
                             "occlusion unknown")
summary = hard_k.groupby(["reason", "type"]).agg(n=("covered", "size"), coverage=("covered", "mean"),
                                               share_overestimated=("overestimated", "mean"))
print(f"Mondrian-by-class intervals (calibrated on clearly visible objects) applied to {len(hard_k)} occluded/truncated test objects")
print(summary.round(3).to_string())
by_reason = hard_k.groupby("reason").agg(coverage=("covered", "mean"), overestimated=("overestimated", "mean"),
                                         n=("covered", "size"))
for reason, r in by_reason.iterrows():
    print(f"{reason:20s}: coverage {r.coverage:.3f} (target {1 - ALPHA:.2f}), "
          f"{r.overestimated:.0%} overestimated (object closer than estimated), n = {r.n}")

### 9.4 What the interval means for AEB

For a braking decision the two ends of the interval have different safety meanings. Underestimating distance makes an object look more urgent than it is (towards **SG-01**, unintended deceleration); overestimating it delays the reaction (towards **SG-02**, omission). With a closing speed $v$, a distance interval $[d_\text{lo}, d_\text{hi}]$ becomes a time-to-collision interval $[d_\text{lo}/v,\ d_\text{hi}/v]$. The table uses the per-class Mondrian thresholds and 50 km/h (13.9 m/s), the upper end of the urban pedestrian sub-scope of the course's AEB item.

In [ ]:
V = 50 / 3.6
rows = []
for k in ["Pedestrian", "Cyclist", "Car"]:
    for d_hat in (10, 20, 30):
        lo_, hi_ = d_hat * np.exp(-q_class[k]), d_hat * np.exp(q_class[k])
        rows.append(dict(cls=k, estimate_m=d_hat, interval_m=f"[{lo_:.1f}, {hi_:.1f}]",
                         ttc_s=f"[{lo_ / V:.2f}, {hi_ / V:.2f}]", ttc_width_s=(hi_ - lo_) / V))
print(f"90 % conformal intervals (per-class Mondrian, clearly visible objects) and TTC at {V:.1f} m/s")
print(pd.DataFrame(rows).round(2).to_string(index=False))

w20 = 20 * (np.exp(q_class["Pedestrian"]) - np.exp(-q_class["Pedestrian"])) / V
print(f"Pedestrian estimated at 20 m: TTC interval width {w20:.2f} s (FTTI: SG-01 0.30 s, SG-02 0.50 s)")

At 20 m the interval spans a TTC range of a few tenths of a second, which is of the same order as the FTTI of the AEB safety goals (300 ms for SG-01, 500 ms for SG-02). The guarantee behind it is marginal over the calibration distribution: one miss in ten is *allowed*, and Section 9.3 shows that for truncated objects the misses become the rule rather than the exception and are biased towards overestimated distance, the direction that delays braking; largely occluded pedestrians lose some coverage as well. A monocular distance estimate with a conformal interval is therefore useful evidence for the **nominal** case, and the truncated and occluded cases have to be covered by other means (radar or LiDAR range, a dedicated occlusion-aware model, or a conservative reaction to partially visible VRUs).

## In industry

**Work products.** Conformal methods touch three ISO/PAS 8800 work products: the **AI safety requirements** (a coverage requirement per class and distance band is a verifiable statement about the quality of an uncertainty estimate), the **V&V report** of the AI component (empirical coverage and interval width per slice, with the exchangeability argument), and the **operation-phase monitoring plan** (coverage audits on labelled field samples, and interval width or set size as a runtime monitor signal). In SOTIF terms, the conditions under which coverage fails, such as truncation and occlusion above, are triggering conditions that go into the triggering-condition analysis with a mitigation.

**Who produces and who reads it.** The perception ML engineer computes the intervals and coverage; the safety engineer formulates the requirement and the conditions under which it is *not* claimed; the system architect decides what the planner or AEB arbitration does with a wide interval; the assessor reviews the argument.

**A verifiable requirement looks like this** (illustrative): *"For pedestrians estimated within 40 m, the 90 % distance interval shall contain the true distance with probability ≥ 0.90 in each predicted distance band, demonstrated on an independent test set split by drive. The guarantee is not claimed for objects more than 30 % truncated or largely occluded; for those, the range is taken from the radar channel."*

**What reviewers and assessors check**

- The exchangeability argument: calibration data come from the same ODD and sensor configuration as operation, are split by drive, and were not used for training or model selection.
- $\alpha$, the score function and the Mondrian groups are stated and justified, and the groups are computable at runtime (predicted class and band, never true values).
- Each group has enough calibration samples (at least $1/\alpha - 1$ for a finite threshold, in practice several hundred) and the test coverage is reported with a binomial confidence interval.
- The excluded conditions are listed explicitly, with their mitigation and their place in the SOTIF triggering-condition analysis.
- Operation KPIs: rolling empirical coverage on labelled audit samples per slice, the distribution of interval widths, and the trigger for recalibration (or adaptive conformal) when coverage drifts.

## Key takeaways

1. Split conformal prediction turns any model and a held-out calibration set into **prediction sets/intervals with a finite-sample, distribution-free coverage guarantee** — it needs exchangeability, not a correct model.
2. The threshold is the $\lceil (n+1)(1-\alpha) \rceil / n$ empirical quantile of the calibration scores (identical to `avsafety.uq.conformal_threshold`); **LAC** gives the smallest sets, **APS** gives adaptive ones.
3. **CQR** carries the guarantee to regression with intervals that adapt to local noise; here it matches the naive band's marginal coverage with a *smaller* average width and far better conditional coverage, keeping far-object coverage at target where the constant band fails.
4. Marginal coverage can hide poor **per-group (class-conditional)** coverage; **Mondrian** conformal restores it by calibrating within each group.
5. **Distribution shift breaks exchangeability** and silently lowers coverage; a *known covariate* shift is repaired by **weighted** conformal, an *unknown time-varying* shift by **adaptive (ACI)** conformal — but a change in $P(Y\mid X)$ is beyond covariate reweighting.
6. On **real KITTI data**, a monocular distance estimate conformalised with a log-residual score keeps coverage much more even across distance than a constant ± band; Mondrian calibration makes the guarantee hold per class; and objects truncated at the image border, which break exchangeability, lose most of their coverage, with the misses biased towards overestimated distance.
7. The prediction **set/interval** is a ready-made safety signal: its size drives conservative behaviour or handover, and its guarantee and failure mode slot directly into an **ISO/PAS 8800 / SOTIF** assurance argument.

## Self-assessment

1. What exactly does the $1-\alpha$ guarantee promise, and what does it *not* promise about any individual prediction?
2. Why can the split-conformal threshold use $\lceil (n+1)(1-\alpha) \rceil / n$ rather than simply the $(1-\alpha)$ quantile? What happens as $n \to \infty$?
3. A fleet log shows 90% marginal coverage but only 78% for the cyclist class. Which conformal variant fixes this, and what is the cost?
4. After a sensor degrades, coverage drops from 0.90 to 0.6 although the model and threshold are unchanged. Which assumption failed, and which two methods can recover coverage — and when does *neither* help?
5. A monitor receives the APS set `{pedestrian, cyclist}` for an object. What should the vehicle do, and why is this safer than acting on the top-1 label alone?
6. Why may a Mondrian group be defined by the *predicted* distance band but not by the *true* distance band?

<details>
<summary>Answers</summary>

1. It promises that, averaged over the random draw of calibration and test data, the prediction set contains the true label with probability at least $1-\alpha$ (**marginal** coverage). It does **not** promise coverage for any specific input, class, or region, nor that a particular set is correct — only the long-run/average rate.
2. The finite-sample proof counts ranks among $n$ calibration scores plus the test point ($n+1$ values); the $\lceil (n+1)(1-\alpha)\rceil$-th order statistic guarantees the inequality exactly, with a small upward correction $1/(n+1)$. As $n \to \infty$ the inflation factor $(n+1)/n \to 1$ and the threshold converges to the ordinary $(1-\alpha)$ population quantile.
3. **Mondrian / class-conditional** conformal, which calibrates a separate threshold per class. The cost is **larger sets** for the harder classes (and the need for enough calibration examples in each class to estimate its quantile reliably).
4. **Exchangeability** between calibration and test data failed (a distribution shift). **Weighted conformal** recovers coverage for a *known covariate* shift (change in $P(X)$, same $P(Y\mid X)$); **ACI** recovers it for an *unknown, time-varying* shift online. Neither helps when the shift changes $P(Y\mid X)$ itself — the model's input–label relationship — which is a performance insufficiency, not a reweighting problem.
5. Treat the object conservatively as the most protected possible member — here a **pedestrian** (reduce speed, widen margin), or hand over if the set is large/empty. This is safer because the set is a *calibrated* statement that the truth is one of those classes with guaranteed probability; acting on the top-1 label alone discards that quantified ambiguity exactly when the model is unsure between a pedestrian and a cyclist.
6. The threshold is chosen per group at test time, when only the prediction is known. Grouping by the true distance would require the quantity being estimated; the guarantee then holds conditionally on the predicted band, which is the information the downstream function actually has.
</details>

## References

Verified via Crossref and arXiv on 28 October 2026.

1. V. Vovk, A. Gammerman and G. Shafer, *Algorithmic Learning in a Random World*, Springer, 2005 (2nd ed. 2022). https://doi.org/10.1007/978-3-031-06649-8
2. A. N. Angelopoulos and S. Bates, *Conformal Prediction: A Gentle Introduction*, Foundations and Trends in Machine Learning 16(4), 2023 (also arXiv:2107.07511). https://doi.org/10.1561/2200000101
3. M. Sadinle, J. Lei and L. Wasserman, *Least Ambiguous Set-Valued Classifiers With Bounded Error Levels* (LAC), Journal of the American Statistical Association 114(525), 2019. https://doi.org/10.1080/01621459.2017.1395341
4. Y. Romano, M. Sesia and E. J. Candès, *Classification with Valid and Adaptive Coverage* (APS), NeurIPS 2020. arXiv:2006.02544. https://arxiv.org/abs/2006.02544
5. Y. Romano, E. Patterson and E. J. Candès, *Conformalized Quantile Regression* (CQR), NeurIPS 2019. arXiv:1905.03222. https://arxiv.org/abs/1905.03222
6. J. Lei, M. G'Sell, A. Rinaldo, R. J. Tibshirani and L. Wasserman, *Distribution-Free Predictive Inference for Regression*, Journal of the American Statistical Association 113(523), 2018. https://doi.org/10.1080/01621459.2017.1307116
7. R. J. Tibshirani, R. Foygel Barber, E. J. Candès and A. Ramdas, *Conformal Prediction Under Covariate Shift* (weighted conformal), NeurIPS 2019. arXiv:1904.06019. https://arxiv.org/abs/1904.06019
8. I. Gibbs and E. J. Candès, *Adaptive Conformal Inference Under Distribution Shift* (ACI), NeurIPS 2021. arXiv:2106.00170. https://arxiv.org/abs/2106.00170
9. ISO/PAS 8800:2024, *Road vehicles — Safety and artificial intelligence.*
10. ISO 21448:2022, *Road vehicles — Safety of the intended functionality (SOTIF).*
11. A. Geiger, P. Lenz and R. Urtasun, *Are we ready for autonomous driving? The KITTI vision benchmark suite*, CVPR 2012. https://doi.org/10.1109/CVPR.2012.6248074

---

**Navigation:** ← [Calibration](04_calibration.ipynb) · [Module index](README.md) · [Explainability →](06_explainability.ipynb)

*Further practice:* the author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop).